# Git Commits Scraper Tutorial

This notebook shows how to scrape well-documented commits from a GitHub repository using the GitHub GraphQL API. We'll fetch commits from the JAX repository and create training data for code generation models.

The scraper will:
1. Fetch commits using GitHub's GraphQL API with pagination
2. Filter for well-documented commits (good message length, reasonable diff size)
3. Get full diffs for each commit via REST API
4. Format everything into training examples

## Imports

In [1]:
import json
import os
from typing import Any, Dict, List, Optional

import requests
from dotenv import load_dotenv
from tqdm import tqdm

## Config

We need a GitHub token to access the API. The script filters commits based on message length and diff size to get quality training data.

In [2]:
assert load_dotenv(), "Couldn't load envvars"
GITHUB_TOKEN: Optional[str] = os.getenv("GITHUB_TOKEN")

GITHUB_GRAPHQL_URL: str = "https://api.github.com/graphql"
REPO_OWNER: str = "google"
REPO_NAME: str = "jax"

MAX_DIFF_CHANGES: int = 500  # Maximum additions/deletions for "good" commits
MIN_COMMIT_MESSAGE_LENGTH: int = 10  # Minimum commit message length
NUM_COMMITS_TO_FETCH: int = 25
MAX_COMMITS_PER_PAGE: int = 25

OUTPUT_FILENAME: str = f"data/{NUM_COMMITS_TO_FETCH}_documented_commits.json"

print(f"Target repo: {REPO_OWNER}/{REPO_NAME}")
print(f"Will fetch: {NUM_COMMITS_TO_FETCH} commits")
print(f"Output: {OUTPUT_FILENAME}")

Target repo: google/jax
Will fetch: 25 commits
Output: data/25_documented_commits.json


## GraphQL Queries

GitHub's GraphQL API is more efficient than REST for fetching multiple commit metadata at once. We'll use two queries: one for commit lists and another for individual commit diffs.

### Understanding the Main Query Structure

The main query follows GitHub's GraphQL schema hierarchy:
- `repository` → `defaultBranchRef` → `target` → `history` → `nodes`

This path represents: "Get the repository → Find its default branch → Get the commit it points to → Access its history → Get the individual commits"

The query uses pagination with `first` and `after` parameters, and includes `pageInfo` to handle large commit histories efficiently.

In [3]:
# Main query to fetch commit metadata with pagination
COMMITS_QUERY: str = """
query GetCommits(
    $owner: String!,
    $name: String!,
    $numCommitsPerPage: Int!,
    $afterCursor: String
) {
  repository(owner: $owner, name: $name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(first: $numCommitsPerPage, after: $afterCursor) {
            nodes {
              oid
              message
              messageHeadline
              messageBody
              committedDate
              url
              additions
              deletions
              changedFiles
              author {
                user {
                  login
                }
                email
                name
              }
              committer {
                user {
                  login
                }
                email
                name
              }
            }
            pageInfo {
              endCursor
              hasNextPage
            }
          }
        }
      }
    }
  }
}
"""

### Key Fields Explained

- **oid**: The commit SHA (object identifier)
- **message/messageHeadline/messageBody**: Different parts of the commit message
- **additions/deletions/changedFiles**: Metrics for filtering commit size
- **author vs committer**: Author wrote the code, committer applied it (can be different)
- **pageInfo**: Contains `endCursor` and `hasNextPage` for pagination

The `... on Commit` syntax is a GraphQL fragment that ensures we're working with a Commit object type.

In [4]:
COMMIT_DIFF_QUERY: str = """
query GetCommitDiff($owner: String!, $name: String!, $oid: String!) {
  repository(owner: $owner, name: $name) {
    object(oid: $oid) {
      ... on Commit {
        oid
        url
      }
    }
  }
}
"""

## GraphQL API Helper

This function handles GraphQL requests to GitHub's API with proper authentication and error handling.

In [5]:
def run_graphql_query(
    query: str, variables: Dict[str, Any], token: str
) -> Dict[str, Any]:
    """
    Executes a GraphQL query against the GitHub API.

    Args:
        query: The GraphQL query string
        variables: Variables for the GraphQL query
        token: GitHub authentication token

    Returns:
        The JSON response from the GraphQL API

    Raises:
        ValueError: If GraphQL API returns errors
        requests.exceptions.RequestException: If HTTP request fails
    """
    headers = {
        "Authorization": f"bearer {token}",
        "Content-Type": "application/json",
    }
    response = requests.post(
        GITHUB_GRAPHQL_URL,
        json={"query": query, "variables": variables},
        headers=headers,
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()
    if "errors" in data:
        raise ValueError(f"GraphQL API errors: {data['errors']}")
    return data

## Test GitHub Authentication

Let's test if your GitHub token works before running the full scraper:

In [6]:
# Test GitHub API authentication with a simple query
def test_github_auth():
    """Test if GitHub token works with a minimal query"""
    if not GITHUB_TOKEN:
        print("❌ No GitHub token found. Please set GITHUB_TOKEN first.")
        return False
    
    # Simple test query to check authentication
    test_query = """
    query {
      viewer {
        login
      }
    }
    """
    
    try:
        result = run_graphql_query(test_query, {}, GITHUB_TOKEN)
        viewer_login = result.get("data", {}).get("viewer", {}).get("login")
        if viewer_login:
            print(f"✅ Authentication successful! Logged in as: {viewer_login}")
            return True
        else:
            print("❌ Authentication failed - no viewer data returned")
            return False
    except Exception as e:
        print(f"❌ Authentication test failed: {e}")
        return False

# Run the test
auth_success = test_github_auth()
if not auth_success:
    print("\n🔧 Please fix your GitHub token before proceeding!")

✅ Authentication successful! Logged in as: neel04


## Getting Commit Diffs

For the actual code changes, we need to use the REST API since GraphQL doesn't return full diffs. This function fetches the unified diff format.

### Why REST API for Diffs?

GitHub's GraphQL API is great for metadata but doesn't include full diff content. The REST API with the `application/vnd.github.v3.diff` accept header returns the actual unified diff format that we need for training data.

In [7]:
def get_commit_diff_via_rest_api(
    owner: str, repo: str, sha: str, token: str
) -> Optional[str]:
    """
    Fetches the diff for a specific commit using GitHub's REST API.

    Args:
        owner: Repository owner
        repo: Repository name
        sha: Commit SHA
        token: GitHub authentication token

    Returns:
        The diff string in unified format, or None if failed
    """
    url = f"https://api.github.com/repos/{owner}/{repo}/commits/{sha}"
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3.diff",
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching diff for commit {sha}: {e}")
        return None

## Getting Commit Diffs

For the actual code changes, we need to use the REST API since GraphQL doesn't return full diffs. This function fetches the unified diff format.

### Why REST API for Diffs?

GitHub's GraphQL API is great for metadata but doesn't include full diff content. The REST API with the `application/vnd.github.v3.diff` accept header returns the actual unified diff format that we need for training data.

In [8]:
def is_well_documented_commit(commit: Dict[str, Any]) -> bool:
    """
    Determines if a commit is well-documented based on our heuristics.

    Args:
        commit: Commit data from GraphQL

    Returns:
        True if the commit meets our documentation criteria
    """
    message = commit.get("message", "")
    additions = commit.get("additions", 0)
    deletions = commit.get("deletions", 0)

    # Check message length
    if len(message.strip()) < MIN_COMMIT_MESSAGE_LENGTH:
        return False

    # Check diff size
    total_changes = additions + deletions
    if total_changes > MAX_DIFF_CHANGES * 2:  # Total of additions + deletions
        return False

    # Additional check: ensure changes aren't too minimal
    if total_changes < 5:  # Skip trivial changes
        return False

    return True

## Extracting Author Info

We want to capture who made the commit for attribution in our training data.

In [9]:
def extract_author_info(commit: Dict[str, Any]) -> List[str]:
    """
    Extracts author information from commit data.

    Args:
        commit: Commit data from GraphQL

    Returns:
        List of author identifiers (GitHub logins when available, emails otherwise)
    """
    authors = []

    # Check commit author
    author = commit.get("author", {})
    if author:
        author_user = author.get("user")
        if author_user and author_user.get("login"):
            authors.append(author_user["login"])
        elif author.get("email"):
            authors.append(author["email"])

    # Check committer (if different from author)
    committer = commit.get("committer", {})
    if committer:
        committer_user = committer.get("user")
        if committer_user and committer_user.get("login"):
            committer_login = committer_user["login"]
            # Only add if different from author
            if committer_login not in authors:
                authors.append(committer_login)
        elif committer.get("email"):
            committer_email = committer["email"]
            if committer_email not in authors:
                authors.append(committer_email)

    return authors if authors else ["unknown"]

## Formatting Diffs

This formats the raw diff with metadata to create a nicely structured output for training.

In [10]:
def format_commit_diff(diff_text: str, commit: Dict[str, Any]) -> str:
    """
    Formats the commit diff with metadata and proper diff formatting.

    Args:
        diff_text: Raw diff text from GitHub API
        commit: Commit metadata

    Returns:
        List containing the formatted diff with metadata
    """
    if not diff_text:
        return "No diff available"

    # Add metadata header
    metadata = [
        f"Commit: {commit.get('oid', 'unknown')}",
        f"Date: {commit.get('committedDate', 'unknown')}",
        f"URL: {commit.get('url', 'unknown')}",
        f"Files changed: {commit.get('changedFiles', 0)}",
        f"Additions: +{commit.get('additions', 0)}, Deletions: -{commit.get('deletions', 0)}",
        "",
    ]

    # Format the diff with proper markdown
    formatted_diff = "```diff\n" + "\n".join(metadata) + diff_text + "\n```"

    return formatted_diff

## Processing to Training Data

This function ties everything together - takes raw commits, filters them, fetches diffs, and creates training examples.

In [11]:
def process_commits_to_training_data(
    commits_nodes: List[Optional[Dict[str, Any]]], owner: str, name: str, token: str
) -> List[Dict[str, Any]]:
    """
    Processes raw commit data from GraphQL into the desired training format.

    Args:
        commits_nodes: List of commit data from GraphQL
        owner: Repository owner
        name: Repository name
        token: GitHub authentication token

    Returns:
        List of training examples in the specified format
    """
    training_data: List[Dict[str, Any]] = []

    for commit in tqdm(commits_nodes):
        if not commit:
            continue

        # Check if commit meets our documentation criteria
        if not is_well_documented_commit(commit):
            continue

        commit_sha = commit.get("oid")
        if not commit_sha:
            continue

        print(f"Processing commit {commit_sha[:8]}...")

        # Get the diff for this commit
        diff_text = get_commit_diff_via_rest_api(owner, name, commit_sha, token)
        if not diff_text:
            print(f"Skipping commit {commit_sha[:8]} - could not fetch diff")
            continue

        # Extract commit message and author info
        commit_message = commit.get("message", "").strip()
        authors = extract_author_info(commit)

        # Format the diff output
        formatted_diff = format_commit_diff(diff_text, commit)

        # Create training example
        training_example = {
            "text_input": commit_message,
            "output": formatted_diff,
            "from_id": authors,
        }

        training_data.append(training_example)
        print(f"Added commit {commit_sha[:8]} to training data")

    return training_data

## Main Execution

Let's break down the main pipeline into logical steps. First, we'll set up the pagination loop to fetch commit metadata.

### Pagination Strategy

GitHub limits API responses, so we fetch commits in batches. We track:
- `current_cursor`: Where to start the next batch  
- `has_next_page`: Whether more commits are available
- `fetched_commits_count`: How many we've collected so far

In [12]:
def fetch_all_commits() -> List[Optional[Dict[str, Any]]]:
    """
    Fetches all commits from the repository using paginated GraphQL queries.

    Returns:
        List of all commit nodes fetched
    """
    all_commits_nodes: List[Optional[Dict[str, Any]]] = []
    current_cursor: Optional[str] = None
    has_next_page: bool = True
    fetched_commits_count: int = 0

    while fetched_commits_count < NUM_COMMITS_TO_FETCH and has_next_page:
        num_to_fetch_this_run = min(
            MAX_COMMITS_PER_PAGE, NUM_COMMITS_TO_FETCH - fetched_commits_count
        )

        print(
            f"Fetching next batch of {num_to_fetch_this_run} commits (cursor: {current_cursor})..."
        )

        variables: Dict[str, Any] = {
            "owner": REPO_OWNER,
            "name": REPO_NAME,
            "numCommitsPerPage": num_to_fetch_this_run,
            "afterCursor": current_cursor,
        }

        try:
            raw_data = run_graphql_query(COMMITS_QUERY, variables, GITHUB_TOKEN)

            repository_data = raw_data.get("data", {}).get("repository", {})
            if not repository_data:
                print("Error: 'repository' field missing in GraphQL response.")
                break

            default_branch = repository_data.get("defaultBranchRef", {})
            if not default_branch:
                print("Error: 'defaultBranchRef' missing in GraphQL response.")
                break

            target = default_branch.get("target", {})
            history = target.get("history", {})

            if not history:
                print("Error: 'history' missing in GraphQL response.")
                break

            new_nodes: List[Optional[Dict[str, Any]]] = history.get("nodes", [])
            all_commits_nodes.extend(new_nodes)
            fetched_commits_count += len(new_nodes)

            page_info: Dict[str, Any] = history.get("pageInfo", {})
            current_cursor = page_info.get("endCursor")
            has_next_page = page_info.get("hasNextPage", False)

            print(
                f"Fetched {len(new_nodes)} commits in this batch. Total fetched so far: {fetched_commits_count}"
            )

            if not has_next_page and fetched_commits_count < NUM_COMMITS_TO_FETCH:
                print("No more commits to fetch from the repository.")
                break

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data from GitHub API: {e}")
            break
        except ValueError as e:
            print(f"GraphQL API Error: {e}")
            break
        except Exception as e:
            print(f"An unexpected error occurred during fetching: {e}")
            break

    return all_commits_nodes

## Putting It All Together

Now let's create a simplified main function that orchestrates all the steps.

In [13]:
def main() -> None:
    """Main function to fetch, process, and save well-documented commits as training data."""

    # Fetch all commits with pagination
    all_commits_nodes = fetch_all_commits()
    
    if not all_commits_nodes:
        print("No commits were fetched successfully.")
        return

    print(f"\nSuccessfully fetched a total of {len(all_commits_nodes)} commits.")
    print("Processing commits into training data...")

    # Process commits into training data
    training_data = process_commits_to_training_data(
        all_commits_nodes, REPO_OWNER, REPO_NAME, GITHUB_TOKEN
    )

    if not training_data:
        print("No training data was generated. Check if commits met the documentation criteria.")
        return

    print(f"Processed {len(training_data)} well-documented commits.")

    # Save the results
    try:
        with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
            json.dump(training_data, f, indent=4, ensure_ascii=False)
        print(f"Successfully saved training data to {OUTPUT_FILENAME}")
        print(f"Total training examples: {len(training_data)}")
    except IOError as e:
        print(f"Error writing data to file {OUTPUT_FILENAME}: {e}")

## Run the Scraper

Let's execute the main function to scrape commits and create our training data.

In [14]:
if __name__ == "__main__":
    main()

Fetching next batch of 25 commits (cursor: None)...
Fetched 25 commits in this batch. Total fetched so far: 25

Successfully fetched a total of 25 commits.
Processing commits into training data...
Fetched 25 commits in this batch. Total fetched so far: 25

Successfully fetched a total of 25 commits.
Processing commits into training data...


  0%|          | 0/25 [00:00<?, ?it/s]

Processing commit e51b63ea...


  4%|▍         | 1/25 [00:00<00:13,  1.74it/s]

Added commit e51b63ea to training data
Processing commit 525a13fb...


...

Added commit 2653c5b5 to training data
Processing commit 7ac8181c...


 92%|█████████▏| 23/25 [00:13<00:00,  2.12it/s]

Added commit 7ac8181c to training data
Processing commit 7f3c9670...


100%|██████████| 25/25 [00:14<00:00,  1.73it/s]

Added commit 7f3c9670 to training data
Processed 22 well-documented commits.
Successfully saved training data to data/25_documented_commits.json
Total training examples: 22


## Analyze Results

Let's examine what we collected and see some sample training examples.

In [ ]:
# Load and analyze the results
if os.path.exists(OUTPUT_FILENAME):
    with open(OUTPUT_FILENAME, "r", encoding="utf-8") as f:
        training_data = json.load(f)
    
    print("📊 Analysis of collected commit data:")
    print(f"Total training examples: {len(training_data)}")
    
    if training_data:
        # Analyze authors
        all_authors = []
        for item in training_data:
            all_authors.extend(item.get('from_id', []))
        unique_authors = set(all_authors)
        
        print(f"Unique contributors: {len(unique_authors)}")
        
        # Show message length distribution
        message_lengths = [len(item['text_input']) for item in training_data]
        print(f"Message length - Min: {min(message_lengths)}, Max: {max(message_lengths)}, Avg: {sum(message_lengths)/len(message_lengths):.1f}")
        
        # Show a sample
        print("\n📝 Sample commit message and diff:")
        sample = training_data[0]
        print(f"Authors: {sample['from_id']}")
        print(f"Message: {sample['text_input'][:200]}...")
        print(f"Diff preview: {sample['output'][:300]}...")
else:
    print(f"❌ Output file {OUTPUT_FILENAME} not found. Run the scraper first.")

📊 Analysis of collected commit data:
Total training examples: 22
Unique contributors: 11
Message length - Min: 62, Max: 1639, Avg: 271.6

📝 Sample commit message and diff:
Authors: ['Google-ML-Automation']
Message: Merge pull request #29901 from harini-sridhar:patch-1

PiperOrigin-RevId: 778456428...
Diff preview: ```diff
Commit: e51b63eaf095edc4a2f5b6c20cfbc04f3c19e69a
Date: 2025-07-02T12:03:20Z
URL: https://github.com/jax-ml/jax/commit/e51b63eaf095edc4a2f5b6c20cfbc04f3c19e69a
Files changed: 1
Additions: +4, Deletions: -41
diff --git a/docs/xla_flags.md b/docs/xla_flags.md
index 24bb8a96c91c..dec2d81e6cf6 10...


## Conclusion

This notebook demonstrates how to:

- Use GitHub's GraphQL API for efficient metadata retrieval
- Handle pagination for large datasets
- Filter commits based on quality heuristics
- Fetch full diffs using REST API
- Format data for machine learning training

The resulting dataset contains commit messages paired with their corresponding code changes, perfect for training models that can generate diffs from natural language descriptions or vice versa.

**Key points:**
- GraphQL is more efficient for fetching metadata in bulk
- REST API is needed for actual diff content
- Quality filtering is crucial for good training data
- Proper attribution helps with model accountability